In [ ]:
import numpy as np
import  pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tabpfn import TabPFNRegressor
import math

def  mape(y_true, y_pred):
    return np.mean(np.abs((y_pred - y_true) / y_true)) * 100


import warnings
warnings.filterwarnings("ignore")

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False 
plt.rcParams['font.serif'] = ['Times New Roman'] 


### （1）TabPFN

In [ ]:
df_data = pd.read_csv('data.csv',sep=',',header=0,index_col=0)

X = df_data.iloc[:,:-5]
y = df_data.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=1/5,    
    random_state=42,  
)


reg = TabPFNRegressor(ignore_pretraining_limits=True)
reg.fit(X_train, y_train)


TabPFN_y_train_pred = reg.predict(X_train)
print('\nTrain mse:',mean_squared_error(TabPFN_y_train_pred,y_train))
print('Train rmse:',np.sqrt(mean_squared_error(TabPFN_y_train_pred,y_train)))
print('Train mae:',mean_absolute_error(TabPFN_y_train_pred,y_train))
print('Train mape:',mape(TabPFN_y_train_pred,y_train))
print('Train R2:',r2_score(y_train,TabPFN_y_train_pred))


TabPFN_y_test_pred = reg.predict(X_test)
print('\nTest mse:',mean_squared_error(TabPFN_y_test_pred,y_test))
print('Test rmse:',np.sqrt(mean_squared_error(TabPFN_y_test_pred,y_test)))
print('Test mae:',mean_absolute_error(TabPFN_y_test_pred,y_test))
print('Test mape:',mape(TabPFN_y_test_pred,y_test))
print('Test R2:',r2_score(y_test,TabPFN_y_test_pred))

threshold = 25
y_train_class = (y_train >= threshold).astype(int)
TabPFN_y_train_pred_class = (TabPFN_y_train_pred >= threshold).astype(int)

y_test_class = (y_test >= threshold).astype(int)
TabPFN_y_test_pred_class = (TabPFN_y_test_pred >= threshold).astype(int)

print('Precision:', precision_score(y_train_class, TabPFN_y_train_pred_class))
print('Recall:', recall_score(y_train_class, TabPFN_y_train_pred_class))
print('F1 Score:', f1_score(y_train_class, TabPFN_y_train_pred_class))

print('Precision:', precision_score(y_test_class, TabPFN_y_test_pred_class))
print('Recall:', recall_score(y_test_class, TabPFN_y_test_pred_class))
print('F1 Score:', f1_score(y_test_class, TabPFN_y_test_pred_class))

### （2）XGBoost

In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import TimeSeriesSplit


def objective(trial):
    param = {
        'booster': 'gbtree',
        'n_estimators': trial.suggest_int('n_estimators', 100,150),
        'learning_rate': trial.suggest_float('learning_rate', 0.1, 0.25),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 30),
        'max_depth': trial.suggest_int('max_depth',10, 20),
        'gamma': trial.suggest_float('gamma', 1e-5, 1e-2, log=True),
        'subsample': trial.suggest_float('subsample', 0.8,1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.25, 0.5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.25, 0.5, log=True),
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'random_state': 0
    }
    
    model = XGBRegressor(**param)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
    return -score

study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=0))


study.optimize(objective, n_trials=100)


best_params = study.best_params
print('best_params:', best_params)


In [ ]:
xgb_reg = XGBRegressor(**best_params, random_state=0)
xgb_reg.fit(X_train,y_train)

XGBoost_y_train_pred = xgb_reg.predict(X_train)
print('\nTrain mse:',mean_squared_error(XGBoost_y_train_pred,y_train))
print('Train rmse:',np.sqrt(mean_squared_error(XGBoost_y_train_pred,y_train)))
print('Train mae:',mean_absolute_error(XGBoost_y_train_pred,y_train))
print('Train mape:',mape(XGBoost_y_train_pred,y_train))
print('Train R2:',xgb_reg.score(X_train,y_train))

XGBoost_y_test_pred = xgb_reg.predict(X_test)
print('\nTest mse:',mean_squared_error(XGBoost_y_test_pred,y_test))
print('Test rmse:',np.sqrt(mean_squared_error(XGBoost_y_test_pred,y_test)))
print('Test mae:',mean_absolute_error(XGBoost_y_test_pred,y_test))
print('Test mape:',mape(XGBoost_y_test_pred,y_test))
print('Test R2:',xgb_reg.score(X_test,y_test))

threshold = 25
y_train_class = (y_train >= threshold).astype(int)
XGBoost_y_train_pred_class = (XGBoost_y_train_pred >= threshold).astype(int)

y_test_class = (y_test >= threshold).astype(int)
XGBoost_y_test_pred_class = (XGBoost_y_test_pred >= threshold).astype(int)

print('Precision:', precision_score(y_train_class, XGBoost_y_train_pred_class))
print('Recall:', recall_score(y_train_class, XGBoost_y_train_pred_class))
print('F1 Score:', f1_score(y_train_class, XGBoost_y_train_pred_class))

print('Precision:', precision_score(y_test_class, XGBoost_y_test_pred_class))
print('Recall:', recall_score(y_test_class, XGBoost_y_test_pred_class))
print('F1 Score:', f1_score(y_test_class, XGBoost_y_test_pred_class))

In [ ]:
import shap
import matplotlib.pyplot as plt

XGBoost_explainer = shap.TreeExplainer(xgb_reg)
XGBoost_shap_values = XGBoost_explainer(X_test)

shap.plots.bar(XGBoost_shap_values, show=False)
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('Chla_XGBoost_shap_importance.svg', format='svg', dpi=600)
plt.close()


shap.plots.bar(XGBoost_shap_values, show=False)

feature_names = X_test.columns.tolist()  
shap_importance_df = pd.DataFrame({'feature': feature_names,'shap_importance': np.abs(XGBoost_shap_values.values).mean(axis=0)})
shap_importance_df = shap_importance_df.sort_values('shap_importance', ascending=False)
print(shap_importance_df)

### (3) RF

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import TimeSeriesSplit

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 150),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': 0
    }

    model = RandomForestRegressor(**param)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
    return -score

study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=0))

study.optimize(objective, n_trials=100)

best_params = study.best_params
print('best_params:', best_params)


In [ ]:
rf_reg = RandomForestRegressor(**best_params, random_state=0)
rf_reg.fit(X_train,y_train)

rf_y_train_pred = rf_reg.predict(X_train)
print('\nTrain mse:',mean_squared_error(rf_y_train_pred,y_train))
print('Train rmse:',np.sqrt(mean_squared_error(rf_y_train_pred,y_train)))
print('Train mae:',mean_absolute_error(rf_y_train_pred,y_train))
print('Train mape:',mape(rf_y_train_pred,y_train))
print('Train R2:',rf_reg.score(X_train,y_train))

rf_y_test_pred = rf_reg.predict(X_test)
print('\nTest mse:',mean_squared_error(rf_y_test_pred,y_test))
print('Test rmse:',np.sqrt(mean_squared_error(rf_y_test_pred,y_test)))
print('Test mae:',mean_absolute_error(rf_y_test_pred,y_test))
print('Test mape:',mape(rf_y_test_pred,y_test))
print('Test R2:',rf_reg.score(X_test,y_test))


threshold = 25
y_train_class = (y_train >= threshold).astype(int)
rf_y_train_pred_class = (rf_y_train_pred >= threshold).astype(int)

y_test_class = (y_test >= threshold).astype(int)
rf_y_test_pred_class = (rf_y_test_pred >= threshold).astype(int)


print('Precision:', precision_score(y_train_class, rf_y_train_pred_class))
print('Recall:', recall_score(y_train_class, rf_y_train_pred_class))
print('F1 Score:', f1_score(y_train_class, rf_y_train_pred_class))


print('Precision:', precision_score(y_test_class, rf_y_test_pred_class))
print('Recall:', recall_score(y_test_class, rf_y_test_pred_class))
print('F1 Score:', f1_score(y_test_class, rf_y_test_pred_class))

In [ ]:
import shap
import matplotlib.pyplot as plt

RF_explainer = shap.TreeExplainer(rf_reg)
RF_shap_values = RF_explainer(X_test)

shap.plots.bar(RF_shap_values, show=False)
plt.title('Random forest Feature Importance')
plt.tight_layout()
plt.savefig('Chla_RF_shap_importance.svg', format='svg', dpi=600)
plt.close()


shap.plots.bar(RF_shap_values, show=False)

feature_names = X_test.columns.tolist() 
shap_importance_df = pd.DataFrame({'feature': feature_names,'shap_importance': np.abs(RF_shap_values.values).mean(axis=0)})
shap_importance_df = shap_importance_df.sort_values('shap_importance', ascending=False)
print(shap_importance_df)